In [ ]:
from typing import List, Optional, Tuple

# Bai toan may hut bui voi ma tran 2 chieu
# DIRTY: ban, CLEAN: sach, 0/1/2... la vi tri hang/cot
Grid = List[List[str]]
Position = Tuple[int, int]
Percept = Tuple[int, int, str]


class VacuumEnvironment:
    def __init__(self, rooms: Grid, start_position: Position):
        self.rooms = [row.copy() for row in rooms]
        self.position = start_position

    def percept(self) -> Percept:
        row, col = self.position
        return row, col, self.rooms[row][col]

    def step(self, action: str) -> None:
        row, col = self.position

        if action == "SUCK":
            self.rooms[row][col] = "CLEAN"
        elif action == "UP" and row > 0:
            self.position = (row - 1, col)
        elif action == "DOWN" and row < len(self.rooms) - 1:
            self.position = (row + 1, col)
        elif action == "LEFT" and col > 0:
            self.position = (row, col - 1)
        elif action == "RIGHT" and col < len(self.rooms[0]) - 1:
            self.position = (row, col + 1)

    def print_state(self) -> None:
        robot_row, robot_col = self.position

        for row_index, row in enumerate(self.rooms):
            line = []
            for col_index, status in enumerate(row):
                if (row_index, col_index) == (robot_row, robot_col):
                    line.append(f"[{status:^5}]")
                else:
                    line.append(f" {status:^5} ")
            print(" ".join(line))
        print(f"Vi tri may hut bui: ({robot_row}, {robot_col})")
        print()


class ModelBasedReflexAgent:
    def __init__(self, rows: int, cols: int):
        # Model la tri nho cua agent ve cac o da quan sat
        self.model = [["UNKNOWN" for _ in range(cols)] for _ in range(rows)]
        self.position: Optional[Position] = None

    def update_model(self, percept: Percept) -> None:
        row, col, status = percept
        self.position = (row, col)
        self.model[row][col] = status

    def all_known_clean(self) -> bool:
        for row in self.model:
            for status in row:
                if status != "CLEAN":
                    return False
        return True

    def find_next_target(self) -> Optional[Position]:
        # Tim o dau tien chua biet hoac dang ban
        for row in range(len(self.model)):
            for col in range(len(self.model[0])):
                if self.model[row][col] != "CLEAN":
                    return row, col
        return None

    def choose_action(self) -> Optional[str]:
        if self.position is None:
            return None

        row, col = self.position

        if self.model[row][col] == "DIRTY":
            return "SUCK"

        if self.all_known_clean():
            return None

        target = self.find_next_target()
        if target is None:
            return None

        target_row, target_col = target
        if row < target_row:
            return "DOWN"
        if row > target_row:
            return "UP"
        if col < target_col:
            return "RIGHT"
        if col > target_col:
            return "LEFT"

        return None


def run_vacuum_agent(initial_rooms: Grid, start_position: Position, max_steps: int = 30) -> None:
    env = VacuumEnvironment(initial_rooms, start_position)
    rows = len(initial_rooms)
    cols = len(initial_rooms[0])
    agent = ModelBasedReflexAgent(rows, cols)

    action_name = {
        "SUCK": "HUT BUI",
        "UP": "DI LEN",
        "DOWN": "DI XUONG",
        "LEFT": "DI SANG TRAI",
        "RIGHT": "DI SANG PHAI",
    }

    print("Trang thai ban dau:")
    env.print_state()

    for step in range(1, max_steps + 1):
        percept = env.percept()
        agent.update_model(percept)
        action = agent.choose_action()

        if action is None:
            print("Tat ca cac o da sach. Agent dung lai.")
            break

        print(f"Buoc {step}: {action_name[action]} ({action})")
        env.step(action)
        env.print_state()


# Ban co the doi ma tran ban dau tai day
initial_rooms = [
    ["DIRTY", "DIRTY"],
    ["CLEAN", "DIRTY"],
]

# Vi tri bat dau: (hang, cot)
start_position = (0, 0)

run_vacuum_agent(initial_rooms, start_position)


Trang thai ban dau:
[DIRTY]  DIRTY 
 CLEAN   DIRTY 
Vi tri may hut bui: (0, 0)

Buoc 1: HUT BUI (SUCK)
[CLEAN]  DIRTY 
 CLEAN   DIRTY 
Vi tri may hut bui: (0, 0)

Buoc 2: DI SANG PHAI (RIGHT)
 CLEAN  [DIRTY]
 CLEAN   DIRTY 
Vi tri may hut bui: (0, 1)

Buoc 3: HUT BUI (SUCK)
 CLEAN  [CLEAN]
 CLEAN   DIRTY 
Vi tri may hut bui: (0, 1)

Buoc 4: DI XUONG (DOWN)
 CLEAN   CLEAN 
 CLEAN  [DIRTY]
Vi tri may hut bui: (1, 1)

Buoc 5: HUT BUI (SUCK)
 CLEAN   CLEAN 
 CLEAN  [CLEAN]
Vi tri may hut bui: (1, 1)

Buoc 6: DI SANG TRAI (LEFT)
 CLEAN   CLEAN 
[CLEAN]  CLEAN 
Vi tri may hut bui: (1, 0)

Tat ca cac o da sach. Agent dung lai.
